Example of building the pair-wise data for one pulmonary fabrosis sample (VUILD96MF)

loading everything prerequest

In [1]:
import sys
sys.path.append("/home/sxr280/Spatialformer_6_10/scripts")
sys.path.append("/home/sxr280/Spatialformer_6_10/utils")
from train import manual_train_fm
import os
import json
import torch
from data_loader import create_dataloader_eval
from datasets import load_from_disk, load_dataset
import pickle
from utils import *
# from utils.utils import GetPairs, get_adj, split_dataset
import json
from tqdm import tqdm
import numpy as np
import argparse

/home/sxr280/miniconda3/envs/spatialformer/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
combined_dataset = load_dataset("TerminatorJ/xenium_pandavid_dataset4", cache_dir = "/home/sxr280/Spatialformer/cache/", num_proc = 8)
combined_dataset_all = concatenate_datasets([combined_dataset["train"], combined_dataset["test"], combined_dataset["validation"]])
index_path = "/home/sxr280/Spatialformer/data/sample_cell_index.pkl"
sample_cell_index = get_index(combined_dataset_all, save_file = index_path)
sample_name = "VUILD96MF"
radius = 30

In [3]:
num_workers = 8 

Generating the dataset

In [4]:
# Filter the dataset for rows corresponding to the current sample_name
sample_index = list(sample_cell_index[sample_name].values())
sample_data = combined_dataset_all.select(sample_index)
sparse_adjmtx,cell_ids = get_adj(sample_data, radius = radius, plot = False)
sample_data = sample_data.select_columns(["Full_Tokens","Gene_Gene_Matrix","Expression","Cell_Ids"])


In [5]:
sample_data

Dataset({
    features: ['Full_Tokens', 'Gene_Gene_Matrix', 'Expression', 'Cell_Ids'],
    num_rows: 50076
})

In [6]:
sample_data = sample_data.map(binary_to_coo_matrix, num_proc = num_workers)

Map (num_proc=8): 100%|██████████| 50076/50076 [01:53<00:00, 441.33 examples/s] 


In [7]:
sample_data = sample_data.remove_columns("Gene_Gene_Matrix")
Pairs = GetPairs(sparse_adjmtx, num_workers = num_workers) #sample_index, (leftdataset, rightdataset), label

100%|██████████| 51/51 [01:57<00:00,  2.30s/it]


ERROR: There are 2 nodes not included
The total number of pairs: 
positive pair:803002
negative pair:803002


In [17]:
# sample_data = sample_data.remove_columns("Gene_Gene_Matrix")

In [8]:
sample_data

Dataset({
    features: ['Full_Tokens', 'Expression', 'Cell_Ids', 'row', 'col', 'data', 'shape'],
    num_rows: 50076
})

In [9]:
# import pdb;pdb.set_trace()
all_left_idxs = list(map(lambda x: x[0],Pairs.all_pairs))
all_right_idxs = list(map(lambda x: x[1],Pairs.all_pairs))
# import pdb; pdb.set_trace()
all_labels = Pairs.all_labels
all_left_dataset = sample_data.select(all_left_idxs)
all_right_dataset = sample_data.select(all_right_idxs)
# import pdb; pdb.set_trace()
left_renamed = all_left_dataset.rename_columns({col: f'left_{col}' for col in all_left_dataset.column_names})
right_renamed = all_right_dataset.rename_columns({col: f'right_{col}' for col in all_right_dataset.column_names})
# import pdb; pdb.set_trace
# Concatenate the two datasets
combined_dataset = concatenate_datasets([left_renamed, right_renamed], axis=1)
combined_dataset = combined_dataset.add_column("Labels", all_labels)
# combined_datasets.append(combined_dataset)
combined_dataset.save_to_disk(f"/home/sxr280/Spatialformer/cache/xenium_{sample_name}_pair", num_proc = 32)

Saving the dataset (37/37 shards): 100%|██████████| 1606004/1606004 [00:31<00:00, 50688.09 examples/s] 
